# CMR: Context Maintenance and Retrieval

The base retrieved-context model of memory search

CMR (Context Maintenance and Retrieval) is the foundational model in jaxcmr. It explains free recall through the interplay of drifting temporal context and item-context associations.

## The Core Mechanism {#core-mechanism}

CMR explains why we remember some items and not others through a single idea: **temporal context**. When you study a list of words, each word updates a mental context that drifts over time. Later, when you try to recall, your current context acts as a cue—items associated with similar contexts are more likely to come to mind.

This produces the three hallmark effects of free recall:

- **Recency**: Recent items share context with the retrieval cue
- **Primacy**: Early items receive stronger encoding (attention-based)
- **Contiguity**: Recalling an item reinstates its context, cueing neighbors

## Mathematical Specification {#math-spec}

### Representations {#representations}

CMR uses three representations:

| Representation | Description |
|----------------|-------------|
| $\mathbf{f}_i$ | Item features (one-hot vectors, size = list length) |
| $\mathbf{c}$ | Temporal context (unit-length vector, size = list length + 1) |
| $M^{FC}$, $M^{CF}$ | Associative memory matrices |

### Memory Initialization {#memory-init}

Before the experiment, two memory matrices exist with pre-experimental associations:

**Item-to-context memory (MFC):**
$$M^{FC}_{pre(ij)} = \begin{cases} 1 - \gamma & \text{if } i=j \\ 0 & \text{otherwise} \end{cases}$$

Each item links to a unique context unit. The parameter $\gamma$ (learning rate) determines how much room remains for new associations.

**Context-to-item memory (MCF):**
$$M^{CF}_{pre(ij)} = \begin{cases} \delta & \text{if } i=j \\ \alpha & \text{otherwise} \end{cases}$$

Here $\delta$ (item support) gives each item a baseline association with its context unit, while $\alpha$ (shared support) provides uniform associations across all items.

### Encoding Phase {#encoding-phase}

When item $i$ is studied:

**1. Retrieve contextual input:**
$$\mathbf{c}^{IN}_i = M^{FC} \mathbf{f}_i$$

**2. Update context (drift):**
$$\mathbf{c}_i = \rho_i \mathbf{c}_{i-1} + \beta_{enc} \mathbf{c}^{IN}_i$$

where $\rho_i$ ensures unit length:
$$\rho_i = \sqrt{1 + \beta_{enc}^2\left[(\mathbf{c}_{i-1} \cdot \mathbf{c}^{IN}_i)^2 - 1\right]} - \beta_{enc}(\mathbf{c}_{i-1} \cdot \mathbf{c}^{IN}_i)$$

**3. Form new associations:**
$$\Delta M^{FC}_{ij} = \gamma \mathbf{f}_i \mathbf{c}_j$$
$$\Delta M^{CF}_{ij} = \phi_i \mathbf{c}_i \mathbf{f}_j$$

**4. Primacy gradient:**

Early items receive stronger associations:
$$\phi_i = \phi_s e^{-\phi_d(i-1)} + 1$$

where $\phi_s$ is the primacy scale and $\phi_d$ is the decay rate.

### Retrieval Phase {#retrieval-phase}

**1. Initialize retrieval context:**

Before recalling, context drifts toward the start-of-list state:
$$\mathbf{c}_{start} = \rho_{N+1} \mathbf{c}_N + \beta_{start} \mathbf{c}_0$$

**2. Compute activations:**
$$\mathbf{a} = M^{CF} \mathbf{c}$$

**3. Apply choice sensitivity:**
$$\mathbf{a}' = \mathbf{a}^{\tau}$$

where $\tau$ (choice sensitivity) sharpens the competition between items.

**4. Convert to probabilities:**
$$P(i) = (1-P_{stop})\frac{a'_i}{\sum_k a'_k}$$

**5. Stop probability:**
$$P_{stop}(j) = \theta_s e^{j\theta_r}$$

where $j$ is the output position, $\theta_s$ is the stop probability scale, and $\theta_r$ is the growth rate.

**6. After recall:**

When item $i$ is recalled, its context is reinstated:
$$\mathbf{c}^{IN} = M^{FC} \mathbf{f}_i$$
$$\mathbf{c}_{new} = \rho \mathbf{c}_{old} + \beta_{rec} \mathbf{c}^{IN}$$

This reinstated context now cues the next recall, producing the contiguity effect.

## Parameters {#parameters}

| Parameter | Symbol | Description |
|-----------|--------|-------------|
| `encoding_drift_rate` | $\beta_{enc}$ | Context integration during study |
| `start_drift_rate` | $\beta_{start}$ | Context shift toward start at retrieval onset |
| `recall_drift_rate` | $\beta_{rec}$ | Context integration after each recall |
| `learning_rate` | $\gamma$ | MFC association strength |
| `primacy_scale` | $\phi_s$ | Initial primacy boost |
| `primacy_decay` | $\phi_d$ | Rate of primacy decay |
| `shared_support` | $\alpha$ | Baseline MCF associations |
| `item_support` | $\delta$ | Self-support in MCF |
| `choice_sensitivity` | $\tau$ | Competition sharpness |
| `stop_probability_scale` | $\theta_s$ | Base stopping probability |
| `stop_probability_growth` | $\theta_r$ | Stopping growth rate |

See [Parameters](parameters.ipynb) for complete documentation.

## Usage {#usage}

In [ ]:
from jaxcmr.models.cmr import CMR

# Define parameters
params = {
    "encoding_drift_rate": 0.5,
    "start_drift_rate": 0.5,
    "recall_drift_rate": 0.5,
    "learning_rate": 0.5,
    "primacy_scale": 2.0,
    "primacy_decay": 0.8,
    "shared_support": 0.05,
    "item_support": 0.25,
    "choice_sensitivity": 0.6,
    "stop_probability_scale": 0.05,
    "stop_probability_growth": 0.2,
    "learn_after_context_update": True,
    "allow_repeated_recalls": False,
}

# Create model for a 16-item list
model = CMR(list_length=16, parameters=params)

# Study phase: present items 1-16
for item_id in range(1, 17):
    model = model.experience(item_id)

# Transition to retrieval
model = model.start_retrieving()

# Get probability of each outcome (stop + 16 items)
probs = model.outcome_probabilities()
print(f"Stop probability: {probs[0]:.3f}")
print(f"Most likely item: {probs[1:].argmax() + 1} (p={probs[1:].max():.3f})")

# Simulate a recall
recalled_item = 16  # Last item (1-indexed)
model = model.retrieve(recalled_item)

# Get next outcome probabilities
probs = model.outcome_probabilities()
print(f"After recalling item 16, most likely next: {probs[1:].argmax() + 1}")

## How Context Creates Memory Effects {#context-effects}

### Recency Effect

At the end of study, context most closely matches the final items. The drift equation ensures that recent context inputs have the largest influence:

```
c_final = ρ₁₅·c₁₄ + β·c¹⁵ᴵᴺ
        = ρ₁₅(ρ₁₄·c₁₃ + β·c¹⁴ᴵᴺ) + β·c¹⁵ᴵᴺ
        = ...
```

Earlier items contribute exponentially less to the final context.

### Contiguity Effect

When you recall item 8, its context is reinstated:

$$\mathbf{c}^{IN} = M^{FC} \mathbf{f}_8$$

This retrieved context contains traces from when item 8 was studied—particularly items 7 and 9, which shared similar contexts. When this reinstated context cues MCF, items 7 and 9 receive high activation.

### Asymmetry

Forward transitions (e.g., 8→9) are more likely than backward transitions (8→7) because:

1. During study, context drifts forward
2. The reinstated context for item 8 resembles the context *after* item 8, not before
3. Thus item 9's stored context has higher similarity than item 7's

## Architecture {#architecture}

CMR combines three swappable components:

```python
CMR(
    list_length=16,
    parameters=params,
    mfc_create_fn=LinearMemory.init_mfc,  # Item → Context
    mcf_create_fn=LinearMemory.init_mcf,  # Context → Item
    context_create_fn=TemporalContext.init,
    termination_policy_create_fn=PositionalTermination,
)
```

Different implementations can be substituted while maintaining the CMR interface. See [Context](context.ipynb) and [Memory](linear_memory.ipynb) for component details.

## References

- Howard, M. W., & Kahana, M. J. (2002). A distributed representation of temporal context. *Journal of Mathematical Psychology*, 46, 269-299.
- Polyn, S. M., Norman, K. A., & Kahana, M. J. (2009). A context maintenance and retrieval model of organizational processes in free recall. *Psychological Review*, 116(1), 129-156.
- Morton, N. W., & Polyn, S. M. (2016). A predictive framework for evaluating models of semantic organization in free recall. *Journal of Memory and Language*, 86, 119-140.